# Crafter DR / MAC changed focal：实时对数图

每次重新运行下面的单元格，都会从 `results/` 重新读取最新 CSV。横轴是 WM 更新次数；MAC 的前 10 轮 warmup 不计入横轴。纵轴为对数刻度，越低越好。依次显示综合、layout 和 inventory changed focal。

每组读取 seed 0–4。在每个 WM 更新位置，粗线为已有 seed 的均值（尚未跑到该位置的 seed 不参与平均）；至少有两个 seed 时，阴影为已有 seed 的最小值到最大值，不是置信区间。DR 同一 seed 若有新旧两份 CSV，读取已完成更新次数更多的一份，不把两次运行混合。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "test/crafter_dr_mac_50/conf/config_mac_crafter_dr_mac_50.yaml").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the project root or test/crafter_dr_mac_50.")
RESULTS = ROOT / "test/crafter_dr_mac_50/results"
METRICS = [
    ("target_val_changed_focal_loss", "Combined changed focal loss"),
    ("target_val_layout_changed_focal_loss", "Layout changed focal loss"),
    ("target_val_inventory_changed_focal_loss", "Inventory changed focal loss"),
]
RUNS = {
    "MAC": {
        seed: (RESULTS / f"mac/mac_balanced_ewc20_epoch10_seed{seed}/mac_crafter_lp_layout_stage_novelty_results.csv",)
        for seed in range(5)
    },
    "DR": {
        seed: (
            RESULTS / f"dr/dr_ewc20_epoch10_seed{seed}/dr_crafter_results.csv",
            RESULTS / f"dr/seed{seed}/dr_crafter_results.csv",
        )
        for seed in range(5)
    },
}

# Re-read every file whenever this cell runs. Only completed CSV rows are plotted.
frames = {}
for method, paths in RUNS.items():
    frames[method] = {}
    for seed, candidates in paths.items():
        available = [(path, pd.read_csv(path)) for path in candidates if path.is_file()]
        if not available:
            continue
        path, frame = max(available, key=lambda item: item[1]["Iter"].max() if not item[1].empty else -1)
        required = {"Iter", *(name for name, _ in METRICS)}
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"{path} is missing CSV columns: {sorted(missing)}")
        frame = frame.copy()
        frame["WM_update"] = frame["Iter"].astype(int) - (10 if method == "MAC" else 0)
        frame = frame.loc[frame["WM_update"] > 0]
        if frame.empty:
            continue  # MAC may still be in its ten warmup rounds.
        frame = frame.drop_duplicates("WM_update", keep="last").set_index("WM_update").sort_index()
        frames[method][seed] = frame
    status = ", ".join(f"seed {seed}: {int(frame.index.max())} updates" for seed, frame in frames[method].items())
    print(f"{method}: {status or 'no CSV yet'}")

colors = {"MAC": "#204d95", "DR": "#27935c"}
fig, axes = plt.subplots(len(METRICS), 1, figsize=(9, 13), sharex=True, constrained_layout=True)
for ax, (column, title) in zip(axes, METRICS):
    for method in ("MAC", "DR"):
        seeds = frames[method]
        if not seeds:
            continue
        color = colors[method]
        for seed, frame in seeds.items():
            values = frame[column].to_numpy(dtype=float)
            if not np.isfinite(values).all() or np.any(values <= 0):
                raise ValueError(f"{method} seed {seed} {column} has a nonpositive or nonfinite value; log scale needs positive losses")
        values = pd.concat({seed: frame[column] for seed, frame in seeds.items()}, axis=1).sort_index()
        updates = values.index.to_numpy()
        counts = values.count(axis=1).to_numpy()
        if np.any(counts >= 2):
            ax.fill_between(updates, values.min(axis=1).to_numpy(), values.max(axis=1).to_numpy(),
                            where=counts >= 2, color=color, alpha=0.22, label=f"{method} seed range")
        ax.plot(updates, values.mean(axis=1).to_numpy(), color=color, linewidth=2.3,
                marker="o", markersize=3, label=f"{method} mean (1–5 available seeds)")
    ax.set_yscale("log")
    ax.set_title(title)
    ax.set_ylabel("Loss (log scale; lower is better)")
    ax.grid(alpha=0.25, which="both")
    ax.legend(frameon=False)
axes[-1].set_xlabel("WM update (MAC warmup excluded)")
fig.suptitle("Crafter DR vs MAC | uniform targets 1–4 × 500", fontsize=12)
output = RESULTS / "changed_focal_log_combined_live_wide.png"
fig.savefig(output, dpi=180)
plt.show()
print(f"Saved: {output}")


## 5 次更新移动平均

先运行上面的原始曲线单元格以重新读取最新 CSV，再运行下面的单元格。先分别对每个 seed 做最近 5 次 WM 更新的移动平均，再计算跨 seed 均值和最小值–最大值阴影；前 4 个点使用当时已有的数据。阴影仍不是置信区间。


In [ ]:
SMOOTH_WINDOW = 5
if "frames" not in globals():
    raise RuntimeError("Run the original plot cell first to load the latest CSV files.")

fig, axes = plt.subplots(len(METRICS), 1, figsize=(9, 13), sharex=True, constrained_layout=True)
for ax, (column, title) in zip(axes, METRICS):
    for method in ("MAC", "DR"):
        seeds = frames[method]
        if not seeds:
            continue
        color = colors[method]
        smoothed = pd.concat(
            {seed: frame[column].rolling(SMOOTH_WINDOW, min_periods=1).mean()
             for seed, frame in seeds.items()},
            axis=1,
        ).sort_index()
        updates = smoothed.index.to_numpy()
        counts = smoothed.count(axis=1).to_numpy()
        if np.any(counts >= 2):
            ax.fill_between(
                updates, smoothed.min(axis=1).to_numpy(), smoothed.max(axis=1).to_numpy(),
                where=counts >= 2, color=color, alpha=0.22, label=f"{method} seed range",
            )
        ax.plot(
            updates, smoothed.mean(axis=1).to_numpy(), color=color, linewidth=2.3,
            marker="o", markersize=3, label=f"{method} smoothed mean",
        )
    ax.set_yscale("log")
    ax.set_title(title)
    ax.set_ylabel("Loss (log scale; lower is better)")
    ax.grid(alpha=0.25, which="both")
    ax.legend(frameon=False)
axes[-1].set_xlabel("WM update (MAC warmup excluded)")
fig.suptitle(
    f"Crafter DR vs MAC | uniform targets 1–4 × 500 | trailing {SMOOTH_WINDOW}-update mean",
    fontsize=12,
)
output_smooth = RESULTS / f"changed_focal_log_combined_smooth{SMOOTH_WINDOW}_live_wide.png"
fig.savefig(output_smooth, dpi=180)
plt.show()
print(f"Saved: {output_smooth}")


## 全量与少量目标验证：DR vs MAC

上图是训练时对目标 1–4（每个目标 500 个样本）的少量验证。下面两张图分别显示 20 个目标的全量验证和目标 1–4 的少量验证，指标都是综合 changed focal loss。每张图的两条线分别是 DR 和 MAC 在相同 WM 更新次数下的 seed 均值；阴影为均值 ± 1 个 seed 间标准差（`ddof=0`），仅在至少两个 seed 有数据时显示。每次运行都会重新读取 CSV。同一 seed 有新旧两份少量验证 CSV 时，选更新次数更多的一份，不混合两次运行。当前全量验证 DR 有 seed 0–4，MAC 只有 seed 0–1，图例会显示实际读取的 seed 数。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "test/crafter_dr_mac_50/conf/config_mac_crafter_dr_mac_50.yaml").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the project root or test/crafter_dr_mac_50.")
RESULTS = ROOT / "test/crafter_dr_mac_50/results"
LOSS = "target_val_changed_focal_loss"
METHODS = ("DR", "MAC")
COLORS = {"DR": "#27935c", "MAC": "#204d95"}
curves = {"Full validation: 20 targets × 500": {}, "Small validation: targets 1–4 × 500": {}}

for method in METHODS:
    full_path = RESULTS / method.lower() / "full20_changed_focal.csv"
    full = pd.read_csv(full_path)
    required = {"Seed", "WM_Update", LOSS}
    if required - set(full.columns):
        raise ValueError(f"{full_path} is missing columns: {sorted(required - set(full.columns))}")
    full = full.loc[full["Seed"].isin(range(5))].drop_duplicates(["Seed", "WM_Update"], keep="last")
    curves["Full validation: 20 targets × 500"][method] = full.pivot(
        index="WM_Update", columns="Seed", values=LOSS,
    ).sort_index()

    small_seeds = {}
    for seed in range(5):
        if method == "DR":
            candidates = (
                RESULTS / f"dr/dr_ewc20_epoch10_seed{seed}/dr_crafter_results.csv",
                RESULTS / f"dr/seed{seed}/dr_crafter_results.csv",
            )
        else:
            candidates = (
                RESULTS / f"mac/mac_balanced_ewc20_epoch10_seed{seed}/mac_crafter_lp_layout_stage_novelty_results.csv",
            )
        available = [(path, pd.read_csv(path)) for path in candidates if path.is_file()]
        if not available:
            continue
        path, frame = max(available, key=lambda item: item[1]["Iter"].max() if not item[1].empty else -1)
        required = {"Iter", LOSS}
        if required - set(frame.columns):
            raise ValueError(f"{path} is missing columns: {sorted(required - set(frame.columns))}")
        updates = frame["Iter"].astype(int) - (10 if method == "MAC" else 0)
        series = pd.Series(frame[LOSS].to_numpy(dtype=float), index=updates.to_numpy())
        series = series.loc[series.index > 0]
        if not series.empty:
            small_seeds[seed] = series.loc[~series.index.duplicated(keep="last")]
    if small_seeds:
        curves["Small validation: targets 1–4 × 500"][method] = pd.concat(small_seeds, axis=1).sort_index()

fig, axes = plt.subplots(2, 1, figsize=(9, 10), sharex=True, constrained_layout=True)
for ax, (title, methods) in zip(axes, curves.items()):
    for method in METHODS:
        wide = methods.get(method)
        if wide is None or wide.empty:
            continue
        data = wide.to_numpy(dtype=float)
        present = ~np.isnan(data)
        if not np.isfinite(data[present]).all() or (data[present] <= 0).any():
            raise ValueError(f"{method} {title} contains a nonpositive or nonfinite loss")
        count = wide.count(axis=1).to_numpy()
        mean = wide.mean(axis=1).to_numpy()
        std = wide.std(axis=1, ddof=0).to_numpy()
        updates = wide.index.to_numpy()
        color = COLORS[method]
        ax.plot(updates, mean, color=color, linewidth=2.3,
                label=f"{method} mean ({len(wide.columns)} seeds)")
        ax.fill_between(updates, mean - std, mean + std,
                        where=(count >= 2) & (mean - std > 0),
                        color=color, alpha=0.22, label=f"{method} ±1 std")
        print(f"{title} | {method}: {len(wide.columns)} seeds, through update {int(wide.index.max())}")
    ax.set_title(title)
    ax.set_ylabel("Changed focal loss (log scale)")
    ax.set_yscale("log")
    ax.grid(alpha=0.25, which="both")
    ax.legend(frameon=False)
axes[-1].set_xlabel("WM update (MAC warmup excluded)")
output = RESULTS / "full_vs_small_changed_focal_mean_std.png"
fig.savefig(output, dpi=180)
plt.show()
print(f"Saved: {output}")


## 全量验证：5 次更新移动平均

先运行上面的“全量与少量目标验证”单元格以重新读取 CSV。对每个 seed 的 20 目标 changed focal loss 分别计算最近 5 次 WM 更新的移动平均（前 4 个点使用已有更新），再逐次计算跨 seed 均值和 ±1 标准差阴影（`ddof=0`）。DR 和 MAC 各一条线；原始结果保留在上一张图。


In [ ]:
FULL_SMOOTH_WINDOW = 5
if "curves" not in globals():
    raise RuntimeError("Run the full-vs-small validation cell first to load the latest CSV files.")

full_curves = curves["Full validation: 20 targets × 500"]
fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
for method in METHODS:
    wide = full_curves.get(method)
    if wide is None or wide.empty:
        continue
    smoothed = wide.rolling(FULL_SMOOTH_WINDOW, min_periods=1).mean()
    count = smoothed.count(axis=1).to_numpy()
    mean = smoothed.mean(axis=1).to_numpy()
    std = smoothed.std(axis=1, ddof=0).to_numpy()
    updates = smoothed.index.to_numpy()
    color = COLORS[method]
    ax.plot(updates, mean, color=color, linewidth=2.3,
            label=f"{method} mean ({len(smoothed.columns)} seeds)")
    ax.fill_between(updates, mean - std, mean + std,
                    where=(count >= 2) & (mean - std > 0),
                    color=color, alpha=0.22, label=f"{method} ±1 std")
ax.set_yscale("log")
ax.set_title(f"Full validation: 20 targets × 500 | trailing {FULL_SMOOTH_WINDOW}-update mean")
ax.set_xlabel("WM update (MAC warmup excluded)")
ax.set_ylabel("Changed focal loss (log scale)")
ax.grid(alpha=0.25, which="both")
ax.legend(frameon=False)
output_smooth = RESULTS / "full20_changed_focal_smoothed_mean_std.png"
fig.savefig(output_smooth, dpi=180)
plt.show()
print(f"Saved: {output_smooth}")
